In [1]:
import math

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

# Utils

In [2]:
def filter_data(data, threshold=0.95):
    min_non_na_count = int(data.shape[0] * threshold)
    cleaned_data = data.dropna(thresh=min_non_na_count, axis=1)
    return cleaned_data

In [3]:
def remove_outliers_iqr(data):
    df = data.copy()
    Q1 = df[df>0].quantile(0.05)
    Q3 = df[df>0].quantile(0.95)
    IQR = Q3 - Q1
    lower_bound = Q1 - 0.5 * IQR
    upper_bound = Q3 + 1.0 * IQR
    df[(df < lower_bound) | (df > upper_bound)] = np.nan
    return df

In [4]:
def train_test_split(data, ratio=0.8):
    train_size = int(math.floor(len(data) * ratio))
    train_data = data[:train_size]
    test_data = data[train_size:]
    return train_data, test_data

In [5]:
def interpolate_rows(df_input):
    df_interpolated = df_input.copy()

    df_interpolated = df_interpolated.fillna(method='ffill', axis=1)
    df_interpolated = df_interpolated.fillna(method='bfill', axis=1)
    return df_interpolated

In [6]:
def train_test_split(data, ratio=0.8):
    train_size = int(math.floor(len(data) * ratio))
    train_data = data[:train_size]
    test_data = data[train_size:]
    return train_data, test_data

In [7]:
def norm_rows(df_input):
    df_scaled = df_input.copy()

    def scale_row(row):
        row_min = row.min()
        row_max = row.max()
        row_range = row_max - row_min

        if pd.isna(row_min):
            return row

        if row_range == 0:
            return row.apply(lambda x: 0.0 if pd.notna(x) else np.nan)
        else:
            return (row - row_min) / row_range

    df_scaled = df_scaled.apply(scale_row, axis=1)

    return df_scaled

# Data

In [9]:
df = pd.read_csv("./data/customer_led_network_revolution/TrialMonitoringDataHH.csv", usecols=["Date and Time of capture", "Location ID", "Parameter"], index_col="Date and Time of capture", engine="c")
df.shape

(300267348, 2)

In [10]:
df_pivot = df.pivot_table(index=df.index, columns='Location ID', values='Parameter')
df_pivot.shape

(42480, 8798)

In [11]:
filtered_df = filter_data(df_pivot)
filtered_df.shape

(42480, 4468)

In [12]:
no_outlier_df = remove_outliers_iqr(filtered_df)
no_outlier_df.shape

(42480, 4468)

In [13]:
df_T = no_outlier_df.T

In [14]:
train, test = train_test_split(df_T)
train, val = train_test_split(train, 0.85)

In [15]:
train.shape, val.shape, test.shape

((3037, 42480), (537, 42480), (894, 42480))

In [16]:
imputed_train = interpolate_rows(train)
imputed_val = interpolate_rows(val)
imputed_test = interpolate_rows(test)

print(imputed_train.isnull().sum().sum(), imputed_test.isnull().sum().sum())

C:\Users\Arne\AppData\Local\Temp\ipykernel_17184\3869416012.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_interpolated = df_interpolated.fillna(method='ffill', axis=1)
C:\Users\Arne\AppData\Local\Temp\ipykernel_17184\3869416012.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_interpolated = df_interpolated.fillna(method='bfill', axis=1)


0 0


In [17]:
norm_train = norm_rows(imputed_train)
norm_val = norm_rows(imputed_val)
norm_test = norm_rows(imputed_test)

print(norm_train.min().min(), norm_train.max().max())
print(norm_val.min().min(), norm_val.max().max())
print(norm_test.min().min(), norm_test.max().max())

0.0 1.0
0.0 1.0
0.0 1.0


In [22]:
norm_train.shape

(3037, 42480)

In [24]:
seq_train = norm_train.iloc[: , :336]
seq_train.shape

(3037, 336)

In [ ]:
norm_train.to_csv("./data/customer_led_network_revolution/preprocessed/train.csv")
norm_val.to_csv("./data/customer_led_network_revolution/preprocessed/val.csv")
norm_test.to_csv("./data/customer_led_network_revolution/preprocessed/test.csv")

# Time Data

In [25]:
train_time = pd.to_datetime(norm_train.columns, format='%d/%m/%Y %H:%M:%S')
train_con = time_fe(norm_train, train_time, 336)
train_con.shape

(3037, 336, 4)